In [7]:
# NOTEBOOK OBJETIVO 2 — UNET vs RF (mapa MT na máscara C10) + área + concordância
# cole célula a célula no Colab (GPU ou CPU tanto faz; é I/O pesado)

# =========================
# CÉLULA 1 — Setup caminhos
# =========================
from pathlib import Path
import os, json

# entradas (ajuste se teus nomes/pastas forem diferentes)
UNET_FULL = Path("/content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full.tif")
UNET_C10  = Path("/content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_c10.tif")   # 0/1/255
RF_MAP    = Path("/content/drive/MyDrive/GEE_Exports/rf_milho_mask1_mt_2023_c10.tif")      # 0/1 na máscara (fora = 0 ou nodata)

# saídas
OUT_DIR   = Path("/content/drive/MyDrive/unet_preds_mt2023_v1/obj2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RF_ON_UNET = OUT_DIR / "rf_on_unetgrid.tif"  # RF reamostrado para a grade do UNET_C10 (0/1/255)
SUMMARY_JSON = OUT_DIR / "area_and_agreement_unet_vs_rf_on_unetgrid.json"

print("UNET_FULL:", UNET_FULL, "exists?", UNET_FULL.exists())
print("UNET_C10 :", UNET_C10,  "exists?", UNET_C10.exists())
print("RF_MAP   :", RF_MAP,    "exists?", RF_MAP.exists())
print("OUT_DIR  :", OUT_DIR)
print("RF_ON_UNET:", RF_ON_UNET)
print("SUMMARY_JSON:", SUMMARY_JSON)

UNET_FULL: /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full.tif exists? True
UNET_C10 : /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_c10.tif exists? True
RF_MAP   : /content/drive/MyDrive/GEE_Exports/rf_milho_mask1_mt_2023_c10.tif exists? True
OUT_DIR  : /content/drive/MyDrive/unet_preds_mt2023_v1/obj2
RF_ON_UNET: /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/rf_on_unetgrid.tif
SUMMARY_JSON: /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/area_and_agreement_unet_vs_rf_on_unetgrid.json


In [8]:
# ============================================
# CÉLULA 2 — Sanity rápido de metadados/uniques
# ============================================
import rasterio
import numpy as np

def quick_unique(path, n=4096*1024):
    with rasterio.open(path) as ds:
        arr = ds.read(1, window=((0, min(ds.height, 2048)), (0, min(ds.width, 2048))))
        u, c = np.unique(arr, return_counts=True)
        return dict(zip(u.tolist(), c.tolist())), ds.crs, ds.transform, (ds.height, ds.width), ds.nodata

u_unet, crs_u, tr_u, sh_u, nd_u = quick_unique(UNET_C10)
u_rf,   crs_r, tr_r, sh_r, nd_r = quick_unique(RF_MAP)

print("UNET_C10 unique(sample):", u_unet, "| crs:", crs_u, "| shape:", sh_u, "| nodata:", nd_u)
print("RF_MAP   unique(sample):", u_rf,   "| crs:", crs_r, "| shape:", sh_r, "| nodata:", nd_r)

UNET_C10 unique(sample): {255: 4194304} | crs: EPSG:4326 | shape: (48240, 48240) | nodata: 255.0
RF_MAP   unique(sample): {0: 4194304} | crs: EPSG:4326 | shape: (39678, 42334) | nodata: None


In [10]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from tqdm import tqdm

assert UNET_C10.exists() and RF_MAP.exists()

def clip_window_to_ds(win, ds):
    # win pode vir com offsets negativos ou tamanhos ~0
    c0 = int(np.floor(win.col_off))
    r0 = int(np.floor(win.row_off))
    c1 = int(np.ceil(win.col_off + win.width))
    r1 = int(np.ceil(win.row_off + win.height))

    c0 = max(0, c0)
    r0 = max(0, r0)
    c1 = min(ds.width, c1)
    r1 = min(ds.height, r1)

    w = c1 - c0
    h = r1 - r0
    if w <= 0 or h <= 0:
        return None
    return Window(c0, r0, w, h)

with rasterio.open(UNET_C10) as u, rasterio.open(RF_MAP) as r:
    profile = u.profile.copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        compress="LZW",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        nodata=255,
        BIGTIFF="YES",
    )

    with rasterio.open(RF_ON_UNET, "w", **profile) as out:
        for _, win in tqdm(list(u.block_windows(1)), desc="rf -> unet grid"):
            ublk = u.read(1, window=win)          # 0/1/255
            valid = (ublk != 255)

            # destino float p/ reproject
            dst = np.full((win.height, win.width), np.nan, dtype=np.float32)
            win_transform = u.window_transform(win)

            # bounds do bloco na CRS EPSG:4326
            left, top = (win_transform * (0, 0))
            right, bottom = (win_transform * (win.width, win.height))
            xmin, xmax = (min(left, right), max(left, right))
            ymin, ymax = (min(bottom, top), max(bottom, top))

            # janela no RF (pode cair fora)
            try:
                rwin = r.window(xmin, ymin, xmax, ymax)
            except Exception:
                rwin = None

            rwin = clip_window_to_ds(rwin, r) if rwin is not None else None

            if rwin is not None:
                rarr = r.read(1, window=rwin)
                if rarr.size > 0 and rarr.shape[0] > 0 and rarr.shape[1] > 0:
                    rtr = r.window_transform(rwin)

                    reproject(
                        source=rarr,
                        destination=dst,
                        src_transform=rtr,
                        src_crs=r.crs,
                        dst_transform=win_transform,
                        dst_crs=u.crs,
                        resampling=Resampling.nearest,
                        src_nodata=r.nodata,   # pode ser None, ok
                        dst_nodata=np.nan
                    )

            # escreve 0/1/255 na grade do UNET
            out_blk = np.full((win.height, win.width), 255, dtype=np.uint8)

            # onde UNET é válido: RF vira 0/1; NaN vira 0
            rf01 = np.nan_to_num(dst, nan=0.0)
            rf01 = (rf01 >= 0.5).astype(np.uint8)

            out_blk[valid] = rf01[valid]
            out.write(out_blk, 1, window=win)

print("OK ✅ RF_ON_UNET:", RF_ON_UNET, "MB=", RF_ON_UNET.stat().st_size/(1024**2))


rf -> unet grid: 100%|██████████| 9025/9025 [02:15<00:00, 66.42it/s] 

OK ✅ RF_ON_UNET: /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/rf_on_unetgrid.tif MB= 23.82414150238037


In [11]:
import rasterio, numpy as np

with rasterio.open(RF_ON_UNET) as ds, rasterio.open(UNET_C10) as u:
    a = ds.read(1, window=((0,2048),(0,2048)))
    uu = u.read(1, window=((0,2048),(0,2048)))
    print("RF_ON_UNET unique(sample):", dict(zip(*np.unique(a, return_counts=True))))
    print("UNET_C10 unique(sample):", dict(zip(*np.unique(uu, return_counts=True))))

RF_ON_UNET unique(sample): {np.uint8(255): np.int64(4194304)}
UNET_C10 unique(sample): {np.uint8(255): np.int64(4194304)}


In [12]:
# ============================================================
# CÉLULA 4 — Área + concordância UNET vs RF (na máscara do UNET)
# - calcula em hectares via reprojeção equal-area (EPSG:5880)
# ============================================================
import rasterio
import numpy as np
from rasterio.warp import calculate_default_transform, reproject, Resampling
from tqdm import tqdm

TARGET_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic (equal-area nacional)

def reproject_to_equal_area(src_path: Path, dst_path: Path, nodata=255):
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )
        profile = src.profile.copy()
        profile.update(
            crs=TARGET_CRS,
            transform=transform,
            width=width,
            height=height,
            nodata=nodata,
            dtype="uint8",
            compress="LZW",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="YES",
            count=1
        )

        with rasterio.open(dst_path, "w", **profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.nearest,
                src_nodata=src.nodata,
                dst_nodata=nodata
            )

UNET_EQ = OUT_DIR / "unet_c10_eqarea.tif"
RF_EQ   = OUT_DIR / "rf_on_unet_eqarea.tif"

reproject_to_equal_area(UNET_C10, UNET_EQ, nodata=255)
reproject_to_equal_area(RF_ON_UNET, RF_EQ, nodata=255)

print("OK ✅ equal-area:", UNET_EQ.name, RF_EQ.name)

OK ✅ equal-area: unet_c10_eqarea.tif rf_on_unet_eqarea.tif


In [13]:
# ============================================================
# CÉLULA 5 — Métricas/áreas finais (ha) + salvar JSON
# ============================================================
import rasterio
import numpy as np
from tqdm import tqdm
import json

with rasterio.open(UNET_EQ) as u, rasterio.open(RF_EQ) as r:
    assert (u.width, u.height) == (r.width, r.height)
    px_area_m2 = abs(u.transform.a * u.transform.e)  # pixel area (m²) em CRS equal-area
    px_area_ha = px_area_m2 / 10000.0

    # acumuladores
    unet1 = unet0 = rf1 = rf0 = mask_area = 0.0
    TP = FP = FN = TN = 0

    for _, win in tqdm(list(u.block_windows(1)), desc="area+agree (C10)"):
        ub = u.read(1, window=win)
        rb = r.read(1, window=win)

        valid = (ub != 255)  # máscara C10 aplicada no UNET_C10
        if not np.any(valid):
            continue

        u0 = (ub == 0) & valid
        u1 = (ub == 1) & valid
        r0 = (rb == 0) & valid
        r1 = (rb == 1) & valid

        n_valid = int(valid.sum())
        mask_area += n_valid * px_area_ha

        unet1 += int(u1.sum()) * px_area_ha
        unet0 += int(u0.sum()) * px_area_ha
        rf1   += int(r1.sum()) * px_area_ha
        rf0   += int(r0.sum()) * px_area_ha

        TP += int((u1 & r1).sum())
        FP += int((u1 & r0).sum())
        FN += int((u0 & r1).sum())
        TN += int((u0 & r0).sum())

summary = {
    "mask_area_ha": float(mask_area),
    "unet_1_area_ha": float(unet1),
    "unet_0_area_ha": float(unet0),
    "rf_1_area_ha": float(rf1),
    "rf_0_area_ha": float(rf0),
    "agreement_pixels": {"TP": TP, "FP": FP, "FN": FN, "TN": TN},
    "notes": {
        "valid_mask": "valid = UNET_C10 != 255",
        "rf_alignment": "RF reamostrado para grade do UNET_C10 (nearest)",
        "area_crs": "EPSG:5880 (equal-area)"
    }
}

SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("OK ✅ salvo:", SUMMARY_JSON)
print(summary)

area+agree (C10): 100%|██████████| 9312/9312 [00:14<00:00, 653.79it/s] 


OK ✅ salvo: /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/area_and_agreement_unet_vs_rf_on_unetgrid.json
{'mask_area_ha': 12217087.88996317, 'unet_1_area_ha': 6141954.806557293, 'unet_0_area_ha': 6075133.083405897, 'rf_1_area_ha': 6270847.965428692, 'rf_0_area_ha': 5946239.924534527, 'agreement_pixels': {'TP': 62136524, 'FP': 8176199, 'FN': 9651760, 'TN': 59895992}, 'notes': {'valid_mask': 'valid = UNET_C10 != 255', 'rf_alignment': 'RF reamostrado para grade do UNET_C10 (nearest)', 'area_crs': 'EPSG:5880 (equal-area)'}}


In [14]:
# ============================================================
# CÉLULA 6 — Arquivos finais p/ QGIS (o que você abre lá)
# ============================================================
print("Abra no QGIS:")
print("1) UNET_C10 (mapa final 0/1 na máscara):", UNET_C10)
print("2) RF_MAP (seu RF original):", RF_MAP)
print("3) RF_ON_UNET (RF na grade do UNET_C10):", RF_ON_UNET)
print("4) summary json (área + concordância):", SUMMARY_JSON)

Abra no QGIS:
1) UNET_C10 (mapa final 0/1 na máscara): /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_c10.tif
2) RF_MAP (seu RF original): /content/drive/MyDrive/GEE_Exports/rf_milho_mask1_mt_2023_c10.tif
3) RF_ON_UNET (RF na grade do UNET_C10): /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/rf_on_unetgrid.tif
4) summary json (área + concordância): /content/drive/MyDrive/unet_preds_mt2023_v1/obj2/area_and_agreement_unet_vs_rf_on_unetgrid.json
